# Thu thập arXiv trên Google Colab — Milestone 1

Chọn **Runtime → Change runtime type → CPU**, rồi chạy các cell theo thứ tự.
Notebook tải đủ các phiên bản nguồn, giữ `.tex/.bib/.bbl`, lấy references và xuất bảng chuẩn cho M1. Nguồn này là đầu vào cho notebook parser M2; chạy crawler xong chưa đồng nghĩa hoàn tất M2.

1. Chọn `CURRENT_MEMBER` của chính mình. Giữ `RUN_MODE="smoke"` để kiểm tra 5 bài với key, Drive và runtime của mình.
2. Nhập S2 key ở ô ẩn, chạy đến hết **validation**. Không ghi key vào source hay chia sẻ output chứa key.
3. Đổi `RUN_MODE="full"`, chạy lại từ cell cấu hình đến validation. Full dùng toàn dải, tự resume và sửa dữ liệu cũ còn thiếu; không bị giới hạn 5 bài của smoke.
4. Sau khi bị ngắt, giữ nguyên thành viên, thư mục Drive và source đã pin; chạy lại notebook để khôi phục. `repair` chỉ sửa những bài đã có, không tải tiếp các ID mới.

| Chọn | Thành viên | MSSV | Dải ID |
| --- | --- | --- | --- |
| DAT | Nguyễn Quốc Đạt | 24120282 | 2412.18848 → 2501.02641 |
| TUAN | Nguyễn Đình Tuấn | 24120237 | 2502.03235 → 2502.08234 |
| TAN | Trần Nguyên Tân | 24120438 | 2407.12839 → 2407.17838 |
| TRI | Dương Phan Minh Trí | 24120470 | 2406.17938 → 2407.02838 |
| TRIET | Trịnh Vỹ Triết | 24120474 | 2406.12938 → 2406.17937 |

Mặc định dùng SSD `/content` và sao lưu nguồn/checkpoint lên Drive mỗi 10 bài. Khi runtime mất đột ngột, tối đa phần công việc chưa vào bản backup thành công có thể phải chạy lại. **Không xóa `backups`, `logs`, `network_state` trên Drive.** Mỗi MSSV chỉ chạy một runtime cùng lúc.

Mỗi thành viên dùng một bản notebook và Drive của mình. Nhiều runtime dùng chung một S2 key vẫn cùng chịu hạn mức API; dùng key được cấp riêng hoặc chạy các lượt dùng chung key lần lượt. Nhịp mặc định là 3 giây cho arXiv; mọi retry/cooldown và cấu hình thực tế được giữ trong số đo.

Source được cố định bằng commit và checksum. Đừng cập nhật source giữa lượt đang chạy. `smoke`, `repair`, `full` lưu riêng từng run ID; chỉ kết luận hoàn tất toàn dải khi validation của **full** đạt.


In [ ]:
import math
import os
import re
import sys
from pathlib import Path

RUN_VALIDATED = False  # Chạy lại cấu hình sẽ vô hiệu validation của run trước.

CURRENT_MEMBER = "DAT"  # @param ["DAT", "TUAN", "TAN", "TRI", "TRIET"]
USE_GOOGLE_DRIVE = True  # @param {type:"boolean"}
USE_LOCAL_WORKSPACE = True  # @param {type:"boolean"}
BACKUP_EVERY_PAPERS = 10  # @param {type:"integer"}
RUN_MODE = "smoke"  # @param ["smoke", "repair", "full"]
SMOKE_PAPERS = 5  # @param {type:"integer"}
MAX_PAPERS = SMOKE_PAPERS if RUN_MODE == "smoke" else 0
SOURCE_MODE = "github"  # @param ["github", "bundle"]
SOURCE_BUNDLE_PATH = ""  # @param {type:"string"}
EXPECTED_BUNDLE_SHA256 = "b973469d3505edab4570340c85d7aecb94d0c8b8aa7c47901c9fa0013b20dab6"

GIT_REPO_URL = "https://github.com/NguyenQuocDat06-AI/arxiv-harvester.git"
GIT_BRANCH = "main"  # @param {type:"string"}
GIT_COMMIT = "16fd9d13c98128a3bdd4c03b76b9c970c8c9be5f"  # @param {type:"string"}
EXPECTED_CRAWLER_CODE_HASH = "75a0caa648ac8a0dbb974566bea4c98f32a890dca72823617ab5c3bd431325d3"
UPDATE_SOURCE = True  # @param {type:"boolean"}

HF_REPO_ID = "nquocdat06/arxiv-harvester-data"
HF_REVISION = "main"  # Có thể pin full commit SHA khi khôi phục.
RESTORE_FROM_HF = False  # @param {type:"boolean"}
ENABLE_HF_UPLOAD = False  # @param {type:"boolean"}
AUTO_SYNC_HF = False  # @param {type:"boolean"}
SYNC_INTERVAL_SEC = 600

# Hub checkpoint có thể thiếu corpus TeX/Bib.
# Tắt kiểm tra chỉ khi resume metadata; M2 vẫn cần nguồn riêng.
REQUIRE_TEX_FOR_RESUME = True
RATE_LIMIT_SEC = 3.0
SOURCE_RATE_LIMIT_SEC = 3.0  # @param {type:"number"}
AUTO_RESUME_ARXIV = True  # @param {type:"boolean"}
RECHECK_PREVIOUS_MISSING = True  # @param {type:"boolean"}
S2_BATCH_SIZE = 5  # @param {type:"integer"}
BATCH_SIZE = SMOKE_PAPERS if RUN_MODE == "smoke" else 25  # @param {type:"integer"}
CHECKPOINT_EVERY_PAPERS = 1  # @param {type:"integer"}
S2_RATE_LIMIT_SEC = 5.0  # @param {type:"number"}
S2_MAX_ATTEMPTS = 1  # @param {type:"integer"}
S2_COOLDOWN_SEC = 60.0  # @param {type:"number"}
MAX_RATE_WAIT_SEC = 180.0  # @param {type:"number"}
SOURCE_DOWNLOAD_ATTEMPTS = 3  # @param {type:"integer"}
SOURCE_READ_TIMEOUT_SEC = 90.0  # @param {type:"number"}
SOURCE_MAX_SECONDS = 180.0  # @param {type:"number"}
PROFILE_INTERVAL_SEC = 0.5
LOG_LEVEL = "INFO"  # @param ["INFO", "DEBUG"]
LOG_HEARTBEAT_SEC = 15
LOG_BACKUP_INTERVAL_SEC = 60
MIN_LOCAL_FREE_GB = 2.0
EXPECTED_FULL_PAPERS = 5000

MEMBERS_CONFIG = {
    "DAT": dict(name="Nguyễn Quốc Đạt", member_id="24120282", start_month="2024-12", start_id=18848, end_month="2025-01", end_id=2641),
    "TUAN": dict(name="Nguyễn Đình Tuấn", member_id="24120237", start_month="2025-02", start_id=3235, end_month="2025-02", end_id=8234),
    "TAN": dict(name="Trần Nguyên Tân", member_id="24120438", start_month="2024-07", start_id=12839, end_month="2024-07", end_id=17838),
    "TRI": dict(name="Dương Phan Minh Trí", member_id="24120470", start_month="2024-06", start_id=17938, end_month="2024-07", end_id=2838),
    "TRIET": dict(name="Trịnh Vỹ Triết", member_id="24120474", start_month="2024-06", start_id=12938, end_month="2024-06", end_id=17937),
}
if RUN_MODE not in {"smoke", "repair", "full"} or SOURCE_MODE not in {"bundle", "github"}:
    raise ValueError("Chọn lại RUN_MODE hoặc SOURCE_MODE")
def require_number(name, minimum, integer=False, strict=False):
    value = globals()[name]
    if (isinstance(value, bool) or not isinstance(value, (int, float))
            or not math.isfinite(value) or (integer and not isinstance(value, int))
            or (value <= minimum if strict else value < minimum)):
        raise ValueError(f"{name}: giá trị không hợp lệ")

for name in ("SMOKE_PAPERS", "BACKUP_EVERY_PAPERS", "BATCH_SIZE", "CHECKPOINT_EVERY_PAPERS",
             "S2_BATCH_SIZE", "S2_MAX_ATTEMPTS", "SOURCE_DOWNLOAD_ATTEMPTS", "EXPECTED_FULL_PAPERS"):
    require_number(name, 1, integer=True)
require_number("MAX_PAPERS", 0, integer=True)
for name in ("SOURCE_RATE_LIMIT_SEC", "S2_RATE_LIMIT_SEC"):
    require_number(name, 1)
require_number("RATE_LIMIT_SEC", 3)
require_number("S2_COOLDOWN_SEC", 0)
require_number("LOG_BACKUP_INTERVAL_SEC", 15)
for name in ("PROFILE_INTERVAL_SEC", "LOG_HEARTBEAT_SEC", "MAX_RATE_WAIT_SEC",
             "SOURCE_READ_TIMEOUT_SEC", "SOURCE_MAX_SECONDS", "MIN_LOCAL_FREE_GB"):
    require_number(name, 0, strict=True)
if CURRENT_MEMBER not in MEMBERS_CONFIG:
    raise ValueError(f"CURRENT_MEMBER phải thuộc {list(MEMBERS_CONFIG)}")
if MAX_PAPERS != (SMOKE_PAPERS if RUN_MODE == "smoke" else 0):
    raise ValueError("Chạy lại cell cấu hình; chỉ smoke mới giới hạn số bài.")
if S2_BATCH_SIZE > 500 or LOG_LEVEL not in {"INFO", "DEBUG"}:
    raise ValueError("Kiểm tra S2_BATCH_SIZE (1–500) và LOG_LEVEL (INFO/DEBUG)")
if SOURCE_MODE == "github" and not re.fullmatch(r"[0-9a-f]{40}", GIT_COMMIT):
    raise ValueError("GIT_COMMIT cần là full SHA đã kiểm tra, dùng bản notebook mới nhất.")
if not re.fullmatch(r"[0-9a-f]{64}", EXPECTED_CRAWLER_CODE_HASH):
    raise ValueError("Thiếu checksum runtime đã kiểm tra.")
if SOURCE_RATE_LIMIT_SEC < 3:
    print("Đang dùng nhịp tải nguồn dưới 3 giây; giữ đúng cấu hình thực tế trong báo cáo.")
if AUTO_SYNC_HF and (not ENABLE_HF_UPLOAD or SYNC_INTERVAL_SEC < 60):
    raise ValueError("Auto-sync cần ENABLE_HF_UPLOAD và chu kỳ >= 60 giây")
if MAX_PAPERS and (RESTORE_FROM_HF or ENABLE_HF_UPLOAD):
    raise ValueError("Smoke dùng thư mục riêng; tắt restore/upload để tránh trộn dữ liệu toàn dải")
if RUN_MODE != "smoke" and not USE_GOOGLE_DRIVE:
    raise ValueError("Full/repair cần Drive để giữ nguồn và checkpoint qua các phiên Colab.")

cfg = MEMBERS_CONFIG[CURRENT_MEMBER].copy()
MEMBER_ID, MEMBER_NAME = cfg["member_id"], cfg["name"]
EXPECTED_RANGE = {key: cfg[key] for key in ("start_month", "start_id", "end_month", "end_id")}
WORKSPACE_DIR = Path("/content/arxiv-harvester")
DRIVE_DATA_ROOT = Path("/content/drive/MyDrive/arxiv-harvester-data")
LOCAL_DATA_ROOT = Path("/content/arxiv-harvester-data")
print(f"Thành viên: {MEMBER_NAME} ({MEMBER_ID})")
print(f"Phạm vi: {EXPECTED_RANGE}; max_papers={MAX_PAPERS or 'toàn dải'}")


## 2. Google Drive và API key

Nhập `S2_API_KEY` trực tiếp vào ô nhập ẩn khi chạy cell. Key chỉ giữ trong bộ nhớ và truyền qua environment cho crawler.

Hugging Face dùng `HF_KEY` trong Secrets khi bật restore/upload; thu thập dữ liệu không cần HF key.


In [ ]:
from google.colab import drive, userdata
from getpass import getpass

if Path("/dev/nvidia0").exists() or os.environ.get("COLAB_TPU_ADDR") or os.environ.get("TPU_NAME"):
    raise RuntimeError("Chọn runtime CPU rồi chạy lại notebook.")
if USE_GOOGLE_DRIVE:
    drive.mount("/content/drive")
DATA_BASE_ROOT = DRIVE_DATA_ROOT if USE_GOOGLE_DRIVE else LOCAL_DATA_ROOT
DATA_ROOT = DATA_BASE_ROOT
NETWORK_STATE_DIR = DATA_BASE_ROOT / "network_state"
if MAX_PAPERS:
    DATA_ROOT = DATA_ROOT / "smoke" / f"{MAX_PAPERS}_papers"
LEGACY_RAW_DIR = DATA_ROOT / "raw" / MEMBER_ID
LEGACY_SHARD_DIR = DATA_ROOT / "shards" / MEMBER_ID
LOCAL_STORAGE_ACTIVE = USE_GOOGLE_DRIVE and globals().get("USE_LOCAL_WORKSPACE", False)
LOCAL_SCOPE_ROOT = LOCAL_DATA_ROOT / "smoke" / f"{MAX_PAPERS}_papers" if MAX_PAPERS else LOCAL_DATA_ROOT
LOCAL_WORK_DIR = LOCAL_SCOPE_ROOT / "workspaces" / MEMBER_ID
BACKUP_DIR = DATA_ROOT / "backups" / MEMBER_ID
RAW_DIR = LOCAL_WORK_DIR / "raw" if LOCAL_STORAGE_ACTIVE else LEGACY_RAW_DIR
SHARD_DIR = LOCAL_WORK_DIR / "shard" if LOCAL_STORAGE_ACTIVE else LEGACY_SHARD_DIR
COURSE_DIR = DATA_ROOT / "course_export" / MEMBER_ID
PATH_CONFIG = (RUN_MODE, MEMBER_ID, MAX_PAPERS)
PATH_MEMBER_CONFIG = (CURRENT_MEMBER, cfg.copy())
PATH_STORAGE_CONFIG = (USE_GOOGLE_DRIVE, globals().get("USE_LOCAL_WORKSPACE", False))
DURABLE_LOG_DIR = DATA_ROOT / "logs" / MEMBER_ID
LOG_DIR = LOCAL_SCOPE_ROOT / "logs" / MEMBER_ID if LOCAL_STORAGE_ACTIVE else DURABLE_LOG_DIR
PATH_LAYOUT = tuple(map(str, (RAW_DIR, SHARD_DIR, BACKUP_DIR, COURSE_DIR, LOG_DIR)))
for directory in (COURSE_DIR, LOG_DIR):
    directory.mkdir(parents=True, exist_ok=True)
if not LOCAL_STORAGE_ACTIVE:
    for directory in (RAW_DIR, SHARD_DIR):
        directory.mkdir(parents=True, exist_ok=True)

def read_secret(name, *env_names):
    for env_name in (name, *env_names):
        value = os.environ.get(env_name, "").strip()
        if value:
            return value
    try:
        return (userdata.get(name) or "").strip()
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError, userdata.TimeoutException):
        return ""

S2_API_KEY = getpass("Nhập S2_API_KEY: ").strip()
if not S2_API_KEY:
    os.environ.pop("S2_API_KEY", None)
    raise ValueError("S2_API_KEY trống; chạy lại cell và nhập key.")
os.environ["S2_API_KEY"] = S2_API_KEY
HF_KEY = read_secret("HF_KEY", "HF_TOKEN") if (ENABLE_HF_UPLOAD or RESTORE_FROM_HF) else ""
if ENABLE_HF_UPLOAD and not HF_KEY:
    raise RuntimeError("Bật quyền Secrets HF_KEY có quyền ghi, hoặc tắt ENABLE_HF_UPLOAD.")
print("Nơi lưu:", DATA_ROOT)
print("Nguồn làm việc:", RAW_DIR)
if LOCAL_STORAGE_ACTIVE:
    print("Nguồn bền vững (archive + HEAD.json):", BACKUP_DIR)
print("Hugging Face upload:", ENABLE_HF_UPLOAD)
print("Semantic Scholar: API key đã nhập")


## 3. Cài source đã kiểm tra

Notebook lấy source từ nhánh `develop` và kiểm tra runtime trước khi chạy.

Để dùng source offline, chọn `SOURCE_MODE="bundle"` và tải lên gói `arxiv_colab_runtime.zip` đi kèm notebook.


In [ ]:
import json
import subprocess
import importlib.metadata
import hashlib
import zipfile
import stat
import tempfile

def git_output(*args):
    return subprocess.check_output(["git", "-C", str(WORKSPACE_DIR), *args], text=True).strip()

def allowed_path(name: str) -> bool:
    path = Path(name)
    return (path.as_posix() == name and not path.is_absolute() and ".." not in path.parts and "\\" not in name and ":" not in name
            and (name in {"pyproject.toml", "requirements.txt", "README.md", "LICENSE"}
                 or (name.startswith("src/") and name.endswith(".py"))
                 or (name.startswith("configs/") and path.suffix in {".json", ".md"})))


def install_bundle(archive_path: str | Path, target_root: str | Path, expected_sha256: str) -> tuple[Path, dict]:
    archive_path, target_root = Path(archive_path), Path(target_root).resolve()
    digest = hashlib.sha256(archive_path.read_bytes()).hexdigest()
    if not expected_sha256 or digest != expected_sha256:
        raise ValueError("Gói source không đúng SHA-256 của notebook")
    with zipfile.ZipFile(archive_path) as archive:
        entries = archive.infolist()
        names = [row.filename for row in entries]
        if len(names) != len(set(names)) or len(names) > 150 or sum(row.file_size for row in entries) > 8 * 1024**2:
            raise ValueError("Gói source có cấu trúc hoặc dung lượng không hợp lệ")
        manifest = json.loads(archive.read("bundle_manifest.json"))
        if manifest.get("format_version") != 1 or manifest.get("runtime_protocol") != 1:
            raise ValueError("Gói source khác phiên bản runtime")
        if set(names) != set(manifest["files"]) | {"bundle_manifest.json"}:
            raise ValueError("Danh sách file không khớp manifest")
        for row in entries:
            if row.filename != "bundle_manifest.json" and not allowed_path(row.filename):
                raise ValueError("File không thuộc source allowlist")
            if stat.S_ISLNK(row.external_attr >> 16) or row.flag_bits & 1:
                raise ValueError("Gói source chứa symlink hoặc file mã hóa")
        contents = {name: archive.read(name) for name in names}
        for name, expected in manifest["files"].items():
            if hashlib.sha256(contents[name]).hexdigest() != expected:
                raise ValueError("Hash file không khớp: " + name)
    target_root.mkdir(parents=True, exist_ok=True)
    destination = target_root / digest[:16]
    if destination.exists():
        if destination.is_symlink() or not destination.resolve().is_relative_to(target_root):
            raise ValueError("Runtime source path is a symlink or outside the target root")
        expected_python = {name for name in manifest["files"] if name.startswith("src/") and name.endswith(".py")}
        actual_python = {p.relative_to(destination).as_posix() for p in (destination / "src").rglob("*.py")}
        if actual_python != expected_python:
            raise ValueError("Runtime source contains unexpected Python files")
        if any(not (destination / name).is_file() or (destination / name).is_symlink()
               or hashlib.sha256((destination / name).read_bytes()).hexdigest() != expected
               for name, expected in manifest["files"].items()):
            raise ValueError("Source đã cài có thay đổi; chọn thư mục source mới")
    else:
        with tempfile.TemporaryDirectory(prefix=".source-", dir=target_root) as temporary:
            staging = Path(temporary).resolve()
            if not staging.is_relative_to(target_root) or not destination.resolve().is_relative_to(target_root):
                raise ValueError("Source destination escapes runtime root")
            unpacked = staging / "runtime"
            for name, data in contents.items():
                target = unpacked / name
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(data)
            unpacked.rename(destination)
    return destination, manifest


if SOURCE_MODE == "bundle":
    archive_path = Path(SOURCE_BUNDLE_PATH) if SOURCE_BUNDLE_PATH else Path("/content/arxiv_colab_runtime.zip")
    if not archive_path.is_file() or (not SOURCE_BUNDLE_PATH and hashlib.sha256(archive_path.read_bytes()).hexdigest() != EXPECTED_BUNDLE_SHA256):
        from google.colab import files
        uploaded = files.upload()
        candidates = [value for name, value in uploaded.items() if name.lower().endswith(".zip")]
        if len(candidates) != 1:
            raise RuntimeError("Chọn đúng một file arxiv_colab_runtime.zip")
        archive_path = Path("/content/arxiv_colab_runtime.zip")
        archive_path.write_bytes(candidates[0])
    WORKSPACE_DIR, source_manifest = install_bundle(archive_path, "/content/arxiv-runtime-sources", EXPECTED_BUNDLE_SHA256)
    SOURCE_COMMIT = None
    SOURCE_PROVENANCE = {"kind": "bundle", "bundle_sha256": EXPECTED_BUNDLE_SHA256,
                         "base_commit": source_manifest["source_base_commit"], "includes_local_changes": True}
else:
    if not WORKSPACE_DIR.exists():
        subprocess.run(["git", "clone", "--branch", GIT_BRANCH, GIT_REPO_URL, str(WORKSPACE_DIR)], check=True)
    else:
        if not (WORKSPACE_DIR / ".git").is_dir():
            raise RuntimeError(f"{WORKSPACE_DIR} đã có nhưng không phải Git checkout.")
        if git_output("remote", "get-url", "origin").rstrip("/") != GIT_REPO_URL.rstrip("/"):
            raise RuntimeError("Checkout thuộc repo khác; chọn WORKSPACE_DIR khác.")
        if git_output("status", "--porcelain", "--untracked-files=no"):
            raise RuntimeError("Checkout có thay đổi local; giữ lại và xử lý trước khi cập nhật/cài source.")
        current_branch = git_output("branch", "--show-current")
        if not GIT_COMMIT and not UPDATE_SOURCE and current_branch != GIT_BRANCH:
            raise RuntimeError("Checkout khác nhánh cấu hình; bật UPDATE_SOURCE hoặc chọn lại cấu hình.")
        if UPDATE_SOURCE and not GIT_COMMIT:
            subprocess.run(["git", "-C", str(WORKSPACE_DIR), "fetch", "origin", GIT_BRANCH], check=True)
            subprocess.run(["git", "-C", str(WORKSPACE_DIR), "checkout", GIT_BRANCH], check=True)
            subprocess.run(["git", "-C", str(WORKSPACE_DIR), "merge", "--ff-only", f"origin/{GIT_BRANCH}"], check=True)
    untracked = git_output("ls-files", "--others", "--exclude-standard", "--", "src")
    if any(name.endswith(".py") for name in untracked.splitlines()):
        raise RuntimeError("Source có Python file ngoài Git; giữ lại rồi chọn checkout sạch.")
    if GIT_COMMIT:
        subprocess.run(["git", "-C", str(WORKSPACE_DIR), "fetch", "origin", GIT_COMMIT], check=True)
        subprocess.run(["git", "-C", str(WORKSPACE_DIR), "checkout", "--detach", GIT_COMMIT], check=True)


    SOURCE_COMMIT = git_output("rev-parse", "HEAD")
    SOURCE_PROVENANCE = {"kind": "github", "commit": SOURCE_COMMIT}

subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(WORKSPACE_DIR), "huggingface_hub"], check=True)
for name in list(sys.modules):
    if name == "src" or name.startswith(("src.crawler", "src.parser")) or name == "crawler" or name.startswith("crawler."):
        sys.modules.pop(name, None)
sys.path.insert(0, str(WORKSPACE_DIR))
os.chdir(WORKSPACE_DIR)
probe = subprocess.run([sys.executable, "-m", "src.crawler.readiness", "--capabilities"],
                       cwd=WORKSPACE_DIR, text=True, capture_output=True)
if probe.returncode != 0:
    raise RuntimeError("Source chưa có runtime mới; chọn SOURCE_MODE='bundle' và dùng gói đi kèm notebook.")
RUNTIME_INFO = json.loads(probe.stdout)
if RUNTIME_INFO.get("protocol_version") != 1 or RUNTIME_INFO.get("checkpoint_schema") != 2:
    raise RuntimeError("Source không đúng phiên bản checkpoint/runtime")
from src.crawler.readiness import code_hash, checkpoint_plan, validate_run, smoke_proof_valid
SOURCE_CODE_HASH = code_hash()
if SOURCE_CODE_HASH != EXPECTED_CRAWLER_CODE_HASH:
    raise RuntimeError("Checksum source khác bản đã kiểm tra; dùng đúng notebook và gói runtime đi kèm.")
if SOURCE_CODE_HASH != RUNTIME_INFO["source_code_hash"]:
    raise RuntimeError("Kernel đang giữ module từ source cũ; restart runtime rồi chạy lại.")
help_text = subprocess.check_output([sys.executable, "-m", "src.crawler.main", "--help"], cwd=WORKSPACE_DIR, text=True)
for flag in ("--resume", "--max-papers", "--existing-only", "--run-id", "--export-course", "--s2-rate-limit-sec", "--s2-max-attempts", "--source-download-attempts", "--source-read-timeout-sec", "--source-max-seconds", "--s2-batch-size", "--checkpoint-every-papers", "--rate-state-dir", "--max-rate-wait-sec", "--source-rate-limit-sec", "--backup-dir", "--backup-every-papers", "--auto-resume-rate-limit", "--recheck-missing"):
    if flag not in help_text:
        raise RuntimeError("CLI còn thiếu " + flag)
PACKAGE_VERSIONS = {name: importlib.metadata.version(name) for name in ("huggingface_hub", "pyarrow", "requests", "psutil", "python-dotenv")}
SOURCE_SETTINGS = (SOURCE_MODE, GIT_REPO_URL, GIT_BRANCH, GIT_COMMIT, EXPECTED_BUNDLE_SHA256, EXPECTED_CRAWLER_CODE_HASH)
print("Source:", SOURCE_PROVENANCE)
print("Code hash:", SOURCE_CODE_HASH)
print("Thư viện:", PACKAGE_VERSIONS)


## 4. Khôi phục và kiểm tra checkpoint

Ưu tiên workspace đang có; trên VM mới sẽ khôi phục chuỗi backup đã xác minh từ Drive, sau đó mới xét thư mục Drive cũ. Không xóa thư mục cũ khi đang chuyển sang SSD. Nếu bật HF restore, checkpoint trên Hub có thể không chứa nguồn; phần nguồn thiếu vẫn phải được tải lại.

Các con số “nguồn tái sử dụng”, “cần kiểm tra references” chỉ mô tả dữ liệu hiện có. Full vẫn tiếp tục ID mới trong cả dải. Không bật hai runtime cùng ghi một MSSV/backup.


In [ ]:
import shutil
import tempfile
import pyarrow.parquet as pq
from huggingface_hub import HfApi, snapshot_download

from src.crawler.drive_backup import prepare_local_workspace, file_hash

if LOCAL_STORAGE_ACTIVE:
    WORKSPACE_RESTORE = prepare_local_workspace(LOCAL_WORK_DIR, BACKUP_DIR, LEGACY_RAW_DIR, LEGACY_SHARD_DIR)
    print("Local workspace:", LOCAL_WORK_DIR, "| restore:", WORKSPACE_RESTORE["source"])
    print("Drive backup:", BACKUP_DIR, "| every", BACKUP_EVERY_PAPERS, "papers")
else:
    WORKSPACE_RESTORE = {"source": "direct_storage"}

from src.crawler.colab_support import mirror_run_artifacts

def check_local_space():
    if LOCAL_STORAGE_ACTIVE:
        free_gb = shutil.disk_usage(LOCAL_WORK_DIR).free / 1024**3
        if free_gb < MIN_LOCAL_FREE_GB:
            raise RuntimeError(f"SSD còn {free_gb:.2f} GiB; cần ít nhất {MIN_LOCAL_FREE_GB} GiB. Giữ backup rồi chuyển runtime.")

def persist_run_artifacts():
    if "RUN_DIR" not in globals() or "RUN_ID" not in globals():
        return
    mirror_run_artifacts(RUN_DIR, SHARD_DIR, RUN_ID,
                         DURABLE_LOG_DIR if LOCAL_STORAGE_ACTIVE else None)

check_local_space()

api = HfApi(token=HF_KEY or False)

def load_checkpoint(shard_dir, check_sources=True):
    shard_dir = Path(shard_dir)
    checkpoint_dir = shard_dir / "_checkpoint"
    state_path = checkpoint_dir / "state.json"
    if not state_path.exists():
        return None
    state = json.loads(state_path.read_text(encoding="utf-8"))
    if state.get("member_id") != MEMBER_ID or state.get("id_range") != EXPECTED_RANGE:
        raise RuntimeError("Checkpoint khác MSSV/khoảng ID đang cấu hình.")
    parts = state.get("parts")
    if not isinstance(parts, dict) or not isinstance(state.get("done_ids"), list):
        raise RuntimeError("Checkpoint sai cấu trúc.")
    for kind in ("publications", "references", "statuses"):
        names = parts.get(kind, [])
        if not isinstance(names, list):
            raise RuntimeError(f"Checkpoint thiếu danh sách parts {kind}.")
        for name in names:
            if not isinstance(name, str) or Path(name).name != name or "/" in name or "\\" in name or not name.endswith(".parquet"):
                raise RuntimeError("Tên part file không hợp lệ.")
            if not (checkpoint_dir / name).is_file():
                raise RuntimeError(f"Thiếu part file: {name}")
            pq.ParquetFile(checkpoint_dir / name)  # Kiểm tra footer đọc được.
    if check_sources:
        plan = checkpoint_plan(shard_dir, RAW_DIR, MEMBER_ID, EXPECTED_RANGE)
        print("Nguồn tái sử dụng:", plan["versions_reusable"],
              "| version cần kiểm tra:", len(plan["source_retry"]),
              "| bài cần xác minh references:", len(plan["references_needing_check"]))
    return state

def restore_checkpoint_from_hf():
    if (SHARD_DIR / "_checkpoint" / "state.json").exists():
        return "Dùng checkpoint local/Drive; không tải đè."
    if any(SHARD_DIR.iterdir()):
        raise RuntimeError("Shard local không rỗng nhưng thiếu state.json; chọn thư mục khác hoặc kiểm tra dữ liệu.")
    info = api.repo_info(repo_id=HF_REPO_ID, repo_type="dataset", revision=HF_REVISION)
    revision = info.sha
    prefix = f"shards/{MEMBER_ID}"
    files = api.list_repo_files(repo_id=HF_REPO_ID, repo_type="dataset", revision=revision)
    if f"{prefix}/_checkpoint/state.json" not in files:
        return "Không có checkpoint remote cho MSSV này; chạy mới."
    with tempfile.TemporaryDirectory(prefix="arxiv-hf-restore-") as temporary:
        snapshot_download(
            repo_id=HF_REPO_ID, repo_type="dataset", revision=revision,
            allow_patterns=[f"{prefix}/_checkpoint/*", f"{prefix}/*.parquet", f"{prefix}/stats.json"],
            local_dir=temporary, token=HF_KEY or False,
        )
        incoming = Path(temporary) / prefix
        load_checkpoint(incoming, check_sources=REQUIRE_TEX_FOR_RESUME)
        shutil.copytree(incoming, SHARD_DIR, dirs_exist_ok=True)
    return f"Đã khôi phục checkpoint tại revision {revision}."

if RESTORE_FROM_HF:
    print(restore_checkpoint_from_hf())
state = load_checkpoint(SHARD_DIR, check_sources=REQUIRE_TEX_FOR_RESUME)
print("Checkpoint:", f"{len(state['done_ids'])} ID đã xử lý theo CLI" if state else "chạy mới")

REPAIR_PLAN = checkpoint_plan(SHARD_DIR, RAW_DIR, MEMBER_ID, EXPECTED_RANGE)
if RUN_MODE == "repair" and not REPAIR_PLAN["exists"]:
    raise RuntimeError("Chế độ repair cần checkpoint trên Drive; chọn đúng thành viên/thư mục.")
if RUN_MODE == "full":
    proof_root = DATA_BASE_ROOT / "smoke" / f"{SMOKE_PAPERS}_papers" / "logs" / MEMBER_ID
    proofs = sorted(proof_root.glob("*/validation.json"), reverse=True)
    FULL_PREFLIGHT = {
        "smoke_validated": any(smoke_proof_valid(p, MEMBER_ID, SOURCE_CODE_HASH, SMOKE_PAPERS) for p in proofs),
        "pending_existing_papers": len(REPAIR_PLAN["pending_papers"]),
    }
    if not FULL_PREFLIGHT["smoke_validated"]:
        print("Chưa có smoke đạt với source hiện tại; full vẫn tiếp tục và được validation sau lượt chạy.")
    if FULL_PREFLIGHT["pending_existing_papers"]:
        print("Full sẽ xử lý lại", FULL_PREFLIGHT["pending_existing_papers"],
              "bài cũ còn thiếu và tiếp tục các ID mới trong dải.")
print("Chế độ:", RUN_MODE, "| bài cũ cần xử lý:", len(REPAIR_PLAN["pending_papers"]))


## 5. Tiến trình và log

Log đầy đủ được hiển thị và ghi vào `crawler.log`; key bị che. Notebook sao lưu log, metadata, HTTP diagnostics và mẫu tài nguyên mỗi 60 giây, sau khi chạy xong và sau validation. Nếu Drive không ghi được, notebook dừng crawler để giữ checkpoint local thay vì chạy tiếp nhiều giờ mà không có bản sao bền vững.


In [ ]:
import queue
import hashlib
import signal
import threading
import time
import uuid
import psutil
from datetime import datetime, timezone

CRAWLER_ENTRY = r'''
"""Add progress logging to the selected crawler checkout."""
import functools
import importlib
import logging
import os
import sys
import time

logging.basicConfig(
    level=os.environ.get("ARXIV_LOG_LEVEL", "INFO"),
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    force=True,
)
logging.getLogger("urllib3").setLevel(logging.WARNING)
logger = logging.getLogger("crawler.progress")
cli = importlib.import_module("src.crawler.main")


def describe(result):
    if isinstance(result, dict) and hasattr(result, "request_stats"):
        statuses = {}
        for row in result.values():
            statuses[row.status] = statuses.get(row.status, 0) + 1
        return f"papers={len(result)}, statuses={statuses}, requests_total={result.request_stats.get('requests', 0)}"
    if isinstance(result, dict):
        fields = ("status", "version_id", "size_before_bytes", "size_after_bytes", "source_dir", "http_status", "attempts", "error_message")
        return ", ".join(f"{key}={result[key]}" for key in fields if key in result)
    if isinstance(result, list):
        parts = [f"records={len(result)}"]
        for name in ("status", "source", "expected_count"):
            if hasattr(result, name):
                parts.append(f"{name}={getattr(result, name)}")
        for name in ("missing_confirmed", "failed_ids"):
            if hasattr(result, name):
                parts.append(f"{name}={len(getattr(result, name))}")
        counts = getattr(result, "request_stats", {})
        if counts:
            parts.extend(f"{name}_total={counts.get(name, 0)}" for name in ("requests", "retries"))
        return ", ".join(parts)
    if isinstance(result, tuple):
        return ", ".join(map(str, result))
    return "OK"


def track(function, label):
    @functools.wraps(function)
    def wrapped(*args, **kwargs):
        if label == "Checkpoint":
            state = args[0].state
            target = f"done={len(state.get('done_ids', []))}, failed={len(state.get('failed', {}))}"
        elif label in ("Publications", "References Parquet"):
            target = str(kwargs.get("output_path", args[1] if len(args) > 1 else ""))
        elif label in ("Discovery", "S2 batch"):
            ids = args[0] if args else kwargs.get("arxiv_ids", [])
            target = f"{len(ids)} ID: {ids[0]} .. {ids[-1]}" if ids else "0 ID"
        else:
            target = str(args[0]) if args else ""
        started = time.perf_counter()
        logger.info("%s bắt đầu: %s", label, target)
        try:
            result = function(*args, **kwargs)
        except BaseException as error:
            logger.error("%s lỗi: %s: %s", label, type(error).__name__, error)
            raise
        logger.info("%s xong: %s | %s | %.2fs", label, target, describe(result), time.perf_counter() - started)
        return result
    return wrapped


for name, label in (
    ("discover_metadata_batch", "Discovery"),
    ("download_and_clean_source", "Source"),
    ("scrape_references_for_paper", "S2 references"),
    ("scrape_references_batch", "S2 batch"),
    ("save_publications_parquet", "Publications"),
    ("save_references_parquet", "References Parquet"),
    ("export_to_course_spec", "Course export"),
):
    function = getattr(cli, name, None)
    if function is not None:
        setattr(cli, name, track(function, label))

checkpoint = getattr(cli, "Checkpoint", None)
if checkpoint is not None and hasattr(checkpoint, "save"):
    checkpoint.save = track(checkpoint.save, "Checkpoint")

try:
    id_range = importlib.import_module("src.crawler.id_range")
except ModuleNotFoundError:
    id_range = None
if id_range is not None and hasattr(id_range, "discover_metadata_batch"):
    id_range.discover_metadata_batch = track(id_range.discover_metadata_batch, "Discovery")

logger.info("Khởi động crawler | log_level=%s", os.environ.get("ARXIV_LOG_LEVEL", "INFO"))
cli.main(sys.argv[1:])
'''


def redact_log(text, secret_values):
    for value in sorted({value for value in secret_values if value}, key=len, reverse=True):
        text = text.replace(value, "[REDACTED]")
    return text


def with_progress_logging(command, directory):
    if command[1:4] != ["-u", "-m", "src.crawler.main"]:
        return command
    entry_path = Path(directory) / "crawler_entry.py"
    entry_path.write_text(CRAWLER_ENTRY, encoding="utf-8")
    return [command[0], "-u", str(entry_path), *command[4:]]


def build_command(export_dir, run_id=None):
    if globals().get("PATH_CONFIG") != (RUN_MODE, MEMBER_ID, MAX_PAPERS):
        raise RuntimeError("Chạy lại cell Drive/đường dẫn sau khi đổi chế độ hoặc thành viên.")
    if "PATH_STORAGE_CONFIG" in globals() and PATH_STORAGE_CONFIG != (USE_GOOGLE_DRIVE, USE_LOCAL_WORKSPACE):
        raise RuntimeError("Chạy lại cell đường dẫn và checkpoint sau khi đổi cách lưu dữ liệu.")
    if "PATH_MEMBER_CONFIG" in globals() and PATH_MEMBER_CONFIG != (CURRENT_MEMBER, cfg):
        raise RuntimeError("Thành viên đã đổi; chạy lại từ cell cấu hình.")
    selected = MEMBERS_CONFIG[CURRENT_MEMBER]
    selected_range = {key: selected[key] for key in ("start_month", "start_id", "end_month", "end_id")}
    if MEMBER_ID != selected["member_id"] or EXPECTED_RANGE != selected_range:
        raise RuntimeError("MSSV hoặc dải ID đang giữ giá trị cũ; chạy lại từ đầu.")
    if MAX_PAPERS != (SMOKE_PAPERS if RUN_MODE == "smoke" else 0):
        raise RuntimeError("Chạy lại cấu hình sau khi đổi chế độ.")
    if "PATH_LAYOUT" in globals() and PATH_LAYOUT != tuple(map(str, (RAW_DIR, SHARD_DIR, BACKUP_DIR, COURSE_DIR, LOG_DIR))):
        raise RuntimeError("Đường dẫn đã đổi; chạy lại cell Drive và checkpoint.")
    if "SOURCE_SETTINGS" in globals():
        settings = (SOURCE_MODE, GIT_REPO_URL, GIT_BRANCH, GIT_COMMIT, EXPECTED_BUNDLE_SHA256, EXPECTED_CRAWLER_CODE_HASH)
        if SOURCE_SETTINGS != settings or code_hash() != SOURCE_CODE_HASH:
            raise RuntimeError("Source đã đổi sau cài đặt; chạy lại từ cell lấy source.")
    if globals().get("ACTIVE_HF_SYNC") is not None and ACTIVE_HF_SYNC.is_alive():
        raise RuntimeError("Upload HF của lượt trước còn chạy; chờ xong trước khi chạy lượt mới.")
    if "check_local_space" in globals():
        check_local_space()
    cmd = [sys.executable, "-u", "-m", "src.crawler.main", "--member-id", MEMBER_ID]
    for name, value in EXPECTED_RANGE.items():
        cmd.extend(["--" + name.replace("_", "-"), str(value)])
    cmd.extend([
        "--raw-dir", str(RAW_DIR), "--output-dir", str(SHARD_DIR),
        "--batch-size", str(BATCH_SIZE), "--rate-limit-sec", str(RATE_LIMIT_SEC),
        "--s2-rate-limit-sec", str(S2_RATE_LIMIT_SEC), "--s2-batch-size", str(S2_BATCH_SIZE),
        "--checkpoint-every-papers", str(CHECKPOINT_EVERY_PAPERS),
        "--rate-state-dir", str(NETWORK_STATE_DIR), "--max-rate-wait-sec", str(MAX_RATE_WAIT_SEC),
        "--s2-max-attempts", str(S2_MAX_ATTEMPTS), "--s2-cooldown-sec", str(S2_COOLDOWN_SEC),
        "--source-download-attempts", str(SOURCE_DOWNLOAD_ATTEMPTS),
        "--source-rate-limit-sec", str(SOURCE_RATE_LIMIT_SEC),
        "--source-read-timeout-sec", str(SOURCE_READ_TIMEOUT_SEC),
        "--source-max-seconds", str(SOURCE_MAX_SECONDS),
        "--profile-interval-sec", str(PROFILE_INTERVAL_SEC),
        "--export-course", "--course-export-dir", str(export_dir),
    ])
    if globals().get("LOCAL_STORAGE_ACTIVE", False):
        cmd.extend(["--backup-dir", str(BACKUP_DIR), "--backup-every-papers", str(BACKUP_EVERY_PAPERS)])
    if load_checkpoint(SHARD_DIR, check_sources=REQUIRE_TEX_FOR_RESUME) is not None:
        cmd.append("--resume")
    elif any(SHARD_DIR.glob("*.parquet")):
        raise RuntimeError("Có working Parquet nhưng thiếu checkpoint; không tự ghi đè để chạy mới.")
    if MAX_PAPERS:
        cmd.extend(["--max-papers", str(MAX_PAPERS)])
    if RUN_MODE == "repair":
        cmd.append("--existing-only")
    if RUN_MODE == "full" and AUTO_RESUME_ARXIV:
        cmd.append("--auto-resume-rate-limit")
    if RUN_MODE == "full" and RECHECK_PREVIOUS_MISSING:
        cmd.append("--recheck-missing")
    if run_id:
        cmd.extend(["--run-id", run_id])
    return cmd

def make_checkpoint_snapshot(destination):
    # Đọc state một lần; các part đã commit không thay đổi.
    state = load_checkpoint(SHARD_DIR, check_sources=False)
    if state is None:
        return False
    target = Path(destination) / "_checkpoint"
    target.mkdir(parents=True, exist_ok=True)
    for kind in ("publications", "references", "statuses"):
        for name in state["parts"].get(kind, []):
            shutil.copyfile(SHARD_DIR / "_checkpoint" / name, target / name)
    (target / "state.json").write_text(json.dumps(state, ensure_ascii=False, indent=2), encoding="utf-8")
    return True

def sync_to_hf(include_outputs=False):
    active = globals().get("ACTIVE_HF_SYNC")
    if active is not None and active.is_alive() and active is not threading.current_thread():
        raise RuntimeError("Upload HF trước chưa xong; chờ trước khi đồng bộ lại.")
    if not ENABLE_HF_UPLOAD:
        return False
    if not HF_KEY:
        raise RuntimeError("Thiếu HF_KEY cho upload.")
    if include_outputs and (not globals().get("RUN_VALIDATED", False)
                            or globals().get("metadata", {}).get("member_id") != MEMBER_ID
                            or globals().get("metadata", {}).get("shard_dir") != str(SHARD_DIR)):
        raise RuntimeError("Kiểm tra kết quả run của đúng MSSV/thư mục trước khi upload bảng cuối.")
    with tempfile.TemporaryDirectory(prefix="arxiv-hf-sync-") as temporary:
        stage = Path(temporary)
        if not make_checkpoint_snapshot(stage):
            return False
        remote = f"shards/{MEMBER_ID}"
        # Upload state sau khi đủ part files.
        if any((stage / "_checkpoint").glob("*.parquet")):
            api.upload_folder(
                repo_id=HF_REPO_ID, repo_type="dataset", folder_path=str(stage / "_checkpoint"),
                path_in_repo=f"{remote}/_checkpoint", allow_patterns=["*.parquet"],
                token=HF_KEY, commit_message=f"Colab checkpoint parts {MEMBER_ID}",
            )
        api.upload_file(
            repo_id=HF_REPO_ID, repo_type="dataset", path_or_fileobj=str(stage / "_checkpoint" / "state.json"),
            path_in_repo=f"{remote}/_checkpoint/state.json", token=HF_KEY,
            commit_message=f"Colab checkpoint state {MEMBER_ID}",
        )
        if include_outputs:
            final_dir = stage / "final"
            final_dir.mkdir()
            for name in ("publications.parquet", "references.parquet", "stats.json"):
                shutil.copyfile(SHARD_DIR / name, final_dir / name)
            if (SHARD_DIR / "crawl_status.parquet").exists():
                shutil.copyfile(SHARD_DIR / "crawl_status.parquet", final_dir / "crawl_status.parquet")
            if (SHARD_DIR / "runs").exists():
                shutil.copytree(SHARD_DIR / "runs", final_dir / "runs", ignore=shutil.ignore_patterns("*.tmp"))
            api.upload_folder(repo_id=HF_REPO_ID, repo_type="dataset", folder_path=str(final_dir),
                              path_in_repo=remote, token=HF_KEY, commit_message=f"Colab final shard {MEMBER_ID}")
            api.upload_folder(repo_id=HF_REPO_ID, repo_type="dataset", folder_path=str(COURSE_DIR),
                              path_in_repo=f"course_export/{MEMBER_ID}", allow_patterns=["papers.parquet", "references.parquet"],
                              token=HF_KEY, commit_message=f"Colab course export {MEMBER_ID}")
    return True

def stop_process(process):
    for sig, timeout in ((signal.SIGINT, 30), (signal.SIGTERM, 10), (signal.SIGKILL, 10)):
        if process.poll() is not None:
            return
        try:
            # Chờ child lưu checkpoint trước khi dừng GNU time.
            try:
                children = psutil.Process(process.pid).children(recursive=True)
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                children = []
            if sig == signal.SIGINT and children:
                for child in reversed(children):
                    try:
                        child.send_signal(sig)
                    except psutil.NoSuchProcess:
                        pass
            else:
                os.killpg(process.pid, sig)
        except ProcessLookupError:
            return
        try:
            process.wait(timeout=timeout)
            return
        except subprocess.TimeoutExpired:
            continue
    raise RuntimeError("Không dừng được tiến trình crawler.")

def run_crawler(command, log_path, time_path):
    global ACTIVE_HF_SYNC
    if not S2_API_KEY:
        raise RuntimeError("Nhập S2_API_KEY ở mục 2 trước khi chạy.")
    logged_command = with_progress_logging(command, Path(log_path).parent)
    wrapped = (["/usr/bin/time", "-v", "-o", str(time_path), *logged_command]
               if Path("/usr/bin/time").exists() else logged_command)
    child_env = os.environ.copy()
    child_env["PYTHONUNBUFFERED"] = "1"
    child_env["PYTHONIOENCODING"] = "utf-8"
    child_env["S2_API_KEY"] = S2_API_KEY
    child_env["ARXIV_LOG_LEVEL"] = LOG_LEVEL
    child_env["PYTHONPATH"] = str(WORKSPACE_DIR) + os.pathsep + child_env.get("PYTHONPATH", "")
    secret_values = (S2_API_KEY, HF_KEY)
    def emit_line(line, log):
        line = redact_log(line, secret_values)
        print(line, end="", flush=True)
        log.write(line)
        log.flush()
    child_env.setdefault("COLAB_RELEASE_TAG", "colab-notebook")
    messages = queue.Queue()
    started = time.perf_counter()
    last_output = time.monotonic()
    if "persist_run_artifacts" in globals():
        persist_run_artifacts()
    process = subprocess.Popen(wrapped, cwd=str(WORKSPACE_DIR), env=child_env,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, encoding="utf-8", errors="replace", bufsize=1,
                               start_new_session=True)
    finished = {}
    def wait_for_exit():
        process.wait()
        finished["perf"] = time.perf_counter()
    waiter = threading.Thread(target=wait_for_exit, daemon=True)
    waiter.start()
    def read_lines():
        try:
            for line in process.stdout:
                messages.put(line)
        finally:
            process.stdout.close()
    reader = threading.Thread(target=read_lines, daemon=True)
    reader.start()
    sync_thread = None
    next_sync = time.monotonic() + SYNC_INTERVAL_SEC
    next_backup = time.monotonic() + globals().get("LOG_BACKUP_INTERVAL_SEC", 60)
    interrupted = False
    def background_sync():
        try:
            if sync_to_hf():
                messages.put("[HF] Đã lưu checkpoint metadata; nguồn vẫn ở RAW_DIR.\n")
        except Exception as exc:
            messages.put(f"[HF] Đồng bộ chưa thành công ({type(exc).__name__}); dữ liệu local/Drive được giữ.\n")
    try:
        with Path(log_path).open("a", encoding="utf-8", buffering=1) as log:
            while process.poll() is None or reader.is_alive() or not messages.empty():
                try:
                    line = messages.get(timeout=0.5)
                except queue.Empty:
                    line = ""
                if line:
                    emit_line(line, log)
                    last_output = time.monotonic()
                elif process.poll() is None and time.monotonic() - last_output >= LOG_HEARTBEAT_SEC:
                    elapsed = time.perf_counter() - started
                    emit_line(f"Tiến trình còn chạy: {elapsed:.0f}s (PID {process.pid}).\n", log)
                    last_output = time.monotonic()
                if time.monotonic() >= next_backup:
                    if "persist_run_artifacts" in globals():
                        persist_run_artifacts()
                    if "check_local_space" in globals():
                        check_local_space()
                    next_backup = time.monotonic() + globals().get("LOG_BACKUP_INTERVAL_SEC", 60)
                if AUTO_SYNC_HF and process.poll() is None and time.monotonic() >= next_sync:
                    if sync_thread is None or not sync_thread.is_alive():
                        sync_thread = threading.Thread(target=background_sync, daemon=True)
                        ACTIVE_HF_SYNC = sync_thread
                        sync_thread.start()
                        next_sync = time.monotonic() + SYNC_INTERVAL_SEC
    except KeyboardInterrupt:
        interrupted = True
        print("\nĐang dừng crawler và chờ lưu checkpoint...")
    finally:
        if process.poll() is None:
            stop_process(process)
        reader.join(timeout=10)
        waiter.join()
        if sync_thread is not None:
            sync_thread.join(timeout=30)
            if sync_thread.is_alive():
                print("Upload HF chưa xong; dữ liệu Drive được giữ. Chờ trước khi chạy hoặc upload lượt mới.")
        with Path(log_path).open("a", encoding="utf-8") as log:
            while not messages.empty():
                line = messages.get_nowait()
                emit_line(line, log)
    return {"returncode": process.wait(), "interrupted": interrupted,
            "effective_command": logged_command, "log_level": LOG_LEVEL,
            "child_wall_sec": round(finished["perf"] - started, 3),
            "runner_wall_sec": round(time.perf_counter() - started, 3)}


## 6. Chạy và lưu kết quả

`smoke` chạy 5 bài trong thư mục riêng; `repair` bổ sung dữ liệu cũ; `full` tiếp tục dải được giao. Crawler dùng chung run ID với notebook. Khi dừng hoặc có lỗi khởi chạy, metadata và log vẫn được lưu; stats của run cũ không được lấy thay.

Nếu VM bị xóa, phiên sau khôi phục lô đã commit gần nhất. Phần sau lô đó cần xử lý lại. Giữ `network_state` và toàn bộ `backups`; chỉ một runtime ghi vào backup của mỗi thành viên.

Full mặc định chờ cooldown arXiv rồi tự tiếp tục trong cùng tiến trình. Checkpoint/backup được lưu trước khi chờ; có thể bấm Stop. S2 mặc định thử một lần mỗi truy vấn để giảm thời gian giữ bước tải nguồn khi S2 đang lỗi; references chưa xác minh vẫn được báo thiếu.


In [ ]:
import traceback

def execute_run():
    global RUN_VALIDATED, RUN_ID, RUN_DIR, RUN_EXPORT_DIR, metadata, result
    RUN_VALIDATED = False
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
    RUN_DIR = LOG_DIR / RUN_ID
    RUN_DIR.mkdir(parents=True, exist_ok=False)
    RUN_EXPORT_DIR = RUN_DIR / "course_export"
    RUN_EXPORT_DIR.mkdir()
    log_path = RUN_DIR / "crawler.log"
    log_path.touch()
    metadata = {
        "run_id": RUN_ID, "member_id": MEMBER_ID, "id_range": EXPECTED_RANGE,
        "source_commit": SOURCE_COMMIT, "source_provenance": SOURCE_PROVENANCE,
        "source_code_hash": SOURCE_CODE_HASH, "package_versions": PACKAGE_VERSIONS,
        "mode": RUN_MODE, "cpu_only": True,
        "preflight": globals().get("FULL_PREFLIGHT") if RUN_MODE == "full" else None,
        "storage": "local_with_drive_backup" if globals().get("LOCAL_STORAGE_ACTIVE", False) else "google_drive" if USE_GOOGLE_DRIVE else "colab_local",
        "backup_dir": str(BACKUP_DIR) if globals().get("LOCAL_STORAGE_ACTIVE", False) else None,
        "max_papers": MAX_PAPERS or None, "raw_dir": str(RAW_DIR), "shard_dir": str(SHARD_DIR),
        "started_at_utc": datetime.now(timezone.utc).isoformat(), "s2_key_source": "direct_input",
        "log_level": LOG_LEVEL, "log_heartbeat_sec": LOG_HEARTBEAT_SEC,
        "logging_adapter_sha256": hashlib.sha256(CRAWLER_ENTRY.encode("utf-8")).hexdigest(),
        "status": "starting",
    }
    def save_metadata():
        temporary = RUN_DIR / "run_metadata.json.tmp"
        temporary.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
        os.replace(temporary, RUN_DIR / "run_metadata.json")
    save_metadata()
    if "persist_run_artifacts" in globals():
        persist_run_artifacts()
    result = {}
    try:
        command = build_command(RUN_EXPORT_DIR, RUN_ID)
        metadata.update(command=command, resumed="--resume" in command, status="running")
        save_metadata()
        with (RUN_DIR / "pip_freeze.txt").open("w", encoding="utf-8") as output:
            subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=output, check=True)
        print("Run:", RUN_ID, "| chế độ:", RUN_MODE)
        print("Lệnh:", " ".join(command))
        print("Log đầy đủ:", log_path, flush=True)
        result = run_crawler(command, log_path, RUN_DIR / "time_v.txt")
        metadata.update(result)
        metadata["status"] = "interrupted" if result["interrupted"] else "completed" if result["returncode"] == 0 else "failed"
        if result["interrupted"]:
            raise RuntimeError("Run đã dừng; checkpoint và nguồn được giữ để resume.")
        if result["returncode"] != 0:
            raise RuntimeError(f"Crawler trả mã {result['returncode']}; xem {log_path}")
    except BaseException as error:
        if metadata["status"] not in {"interrupted", "failed"}:
            metadata["status"] = "interrupted" if isinstance(error, KeyboardInterrupt) else "failed"
        metadata["error_type"] = type(error).__name__
        metadata["error_message"] = redact_log(str(error), (S2_API_KEY, HF_KEY))
        with log_path.open("a", encoding="utf-8") as log:
            log.write(redact_log(traceback.format_exc(), (S2_API_KEY, HF_KEY)))
            log.flush()
        if metadata["error_message"] != str(error):
            raise RuntimeError(metadata["error_message"]) from None
        raise
    finally:
        stats_path = SHARD_DIR / "runs" / RUN_ID / "stats.json"
        metadata["stats_present"] = False
        if stats_path.exists():
            stats = json.loads(stats_path.read_text(encoding="utf-8"))
            if stats.get("run_id") == RUN_ID:
                shutil.copyfile(stats_path, RUN_DIR / "stats.json")
                metadata["stats_present"] = True
                metadata["crawler_status"] = stats.get("status")
                if metadata["status"] == "completed" and stats.get("status") != "completed":
                    metadata["status"] = stats.get("status") or "missing_crawler_status"
                if stats.get("status") == "paused_rate_limit":
                    metadata.update(status="paused_rate_limit", retry_after_sec=stats.get("retry_after_sec"))
        if metadata["status"] == "completed" and not metadata["stats_present"]:
            metadata["status"] = "missing_stats"
        metadata["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
        save_metadata()
        if "persist_run_artifacts" in globals():
            persist_run_artifacts()
    print("CLI kết thúc; chạy cell validation.")
    return result

result = execute_run()


## 7. Validation và xuất bảng chuẩn

Chỉ công bố bảng chuẩn khi schema, nguồn, references, run ID và dải ID đều hợp lệ. Full cần đủ 5.000 ID được giao, gồm bài lưu thành công và ID đã được xác nhận không tồn tại; lỗi mạng không được tính là ID thiếu hợp lệ. `completed_with_errors` cần chạy lại cùng chế độ để xử lý phần còn thiếu.

Giữ `logs/<MSSV>/<run_id>` của **mọi phiên** khi tổng hợp benchmark. Thời gian retry và cooldown thuộc lượt đo thực tế; không sửa lại log để đổi cấu hình hoặc môi trường đo. Các lượt smoke/repair chưa đại diện benchmark toàn dải của cả nhóm.


In [ ]:
RUN_VALIDATED = False

def validate_current_run():
    if globals().get("result", {}).get("returncode") != 0 or result.get("interrupted"):
        raise RuntimeError("Cần một run kết thúc bình thường trước khi validation.")
    if metadata.get("member_id") != MEMBER_ID or metadata.get("shard_dir") != str(SHARD_DIR) or metadata.get("mode") != RUN_MODE:
        raise RuntimeError("Cấu hình đã đổi sau run; chạy lại đúng các cell trước.")
    expected_papers = MAX_PAPERS if RUN_MODE == "smoke" else EXPECTED_FULL_PAPERS if RUN_MODE == "full" else None
    summary = validate_run(SHARD_DIR, RAW_DIR, RUN_EXPORT_DIR, RUN_DIR, MEMBER_ID, RUN_ID, RUN_MODE,
                           expected_range=EXPECTED_RANGE, expected_papers=expected_papers)
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    if metadata.get("crawler_status") == "paused_rate_limit":
        print("Đang nghỉ theo giới hạn dịch vụ; còn khoảng", round(metadata.get("retry_after_sec") or 0), "giây")
        print("Checkpoint được giữ. Giữ thư mục network_state và chạy lại cùng chế độ sau cooldown. Xem http_diagnostics.jsonl trong runs/<run_id>.")
        return False
    if not summary["ready_for_full"]:
        raise RuntimeError("Validation chưa đạt; xem errors và source_audit.retry.jsonl trong RUN_DIR.")
    for name in ("papers.parquet", "references.parquet"):
        temporary = COURSE_DIR / (name + ".tmp")
        shutil.copyfile(RUN_EXPORT_DIR / name, temporary)
        if file_hash(temporary) != file_hash(RUN_EXPORT_DIR / name):
            raise OSError("Bảng chuẩn sao lưu Drive sai checksum; giữ nguyên export theo run ID.")
        os.replace(temporary, COURSE_DIR / name)
    print("Validation đạt | chế độ:", RUN_MODE)
    if RUN_MODE == "repair":
        print("Repair đạt cho các bài đã có; chọn full để tiếp tục toàn dải.")
    elif RUN_MODE == "smoke":
        print("Smoke đạt; đổi RUN_MODE='full' rồi chạy lại từ cell cấu hình.")
    print("Nguồn làm việc:", RAW_DIR)
    if globals().get("LOCAL_STORAGE_ACTIVE", False):
        print("Nguồn bền vững, dùng để restore hoặc đưa vào M2:", BACKUP_DIR)
        print("Giữ HEAD.json và toàn bộ archive/record trong thư mục backup.")
    print("Checkpoint:", SHARD_DIR / "_checkpoint")
    print("Bảng chuẩn:", COURSE_DIR)
    print("Log/run metadata:", RUN_DIR)
    return True

try:
    RUN_VALIDATED = validate_current_run()
finally:
    if "persist_run_artifacts" in globals():
        persist_run_artifacts()


## 8. Đồng bộ Hugging Face

Cần `ENABLE_HF_UPLOAD = True` và HF key. Remote dùng `shards/<MSSV>/` và `course_export/<MSSV>/`. Trước validation chỉ đồng bộ checkpoint metadata. Nguồn TeX/Bib được lưu riêng tại `RAW_DIR`.


In [ ]:
if ENABLE_HF_UPLOAD:
    uploaded = sync_to_hf(include_outputs=bool(globals().get("RUN_VALIDATED", False)))
    print("Đã đồng bộ." if uploaded else "Chưa có checkpoint để đồng bộ.")
    print(f"Dataset: https://huggingface.co/datasets/{HF_REPO_ID}/tree/main/shards/{MEMBER_ID}")
else:
    print("Upload tắt. Kết quả đang được giữ tại:", DATA_ROOT)


## 9. Lấy kết quả và bàn giao cho M2

Mặc định không đóng gói tải xuống vì dữ liệu đã ở Drive. Gói tải xuống bên dưới chứa bảng, checkpoint và log, **không chứa corpus nguồn**.

Khi dùng SSD, corpus nằm bền vững ở `MyDrive/arxiv-harvester-data/backups/<MSSV>/`: giữ `HEAD.json`, toàn bộ `.json` và `.tar.gz` của chuỗi backup. Khôi phục bằng `prepare_local_workspace` như cell 4 để lấy lại `raw` và `shard`, rồi đưa `raw/unzipped` cùng bảng working vào parser M2. Thư mục `raw/<MSSV>` cũ trên Drive có thể chưa bao gồm các bài tải sau khi chuyển sang SSD.


In [ ]:
DOWNLOAD_RESULTS = False  # @param {type:"boolean"}
if DOWNLOAD_RESULTS:
    import tarfile
    from google.colab import files
    archive_path = Path("/content") / f"arxiv-{MEMBER_ID}-{RUN_ID}.tar.gz"
    with tarfile.open(archive_path, "w:gz") as archive:
        archive.add(SHARD_DIR, arcname=f"shards/{MEMBER_ID}")
        if RUN_VALIDATED:
            for name in ("papers.parquet", "references.parquet"):
                archive.add(COURSE_DIR / name, arcname=f"course_export/{MEMBER_ID}/{name}")
        archive.add(RUN_DIR, arcname=f"logs/{MEMBER_ID}/{RUN_ID}")
    files.download(str(archive_path))
else:
    print("Có thể lấy kết quả trực tiếp từ Google Drive hoặc bật DOWNLOAD_RESULTS.")


Tham khảo: [Colab và lưu dữ liệu](https://research.google.com/colaboratory/faq.html),
[Colab Secrets](https://github.com/googlecolab/colabtools/blob/main/google/colab/userdata.py),
[Hub download](https://huggingface.co/docs/huggingface_hub/guides/download),
[Hub upload](https://huggingface.co/docs/huggingface_hub/guides/upload),
[yêu cầu Milestone 1](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/).
